### Conexión AstraDB y Spark session

In [14]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/big-data-final"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# Instalar Java 17
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

# Instalar PySpark y driver
!pip install pyspark==3.5.0 cassandra-driver -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

print("Dependencias instaladas.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 20.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.5.0 which is incompatible.
Dependencias instaladas.


In [15]:
# Rutas del Data Lake
LANDING_PATH = f"{PROJECT_ROOT}/datalake/landing"
BRONZE_PATH = f"{PROJECT_ROOT}/datalake/bronze"
SILVER_PATH = f"{PROJECT_ROOT}/datalake/silver"
GOLD_PATH = f"{PROJECT_ROOT}/datalake/gold"
CHECKPOINT_PATH = f"{PROJECT_ROOT}/datalake/checkpoints"
QUARANTINE_PATH = f"{PROJECT_ROOT}/datalake/quarantine"

# Crear estructura de directorios si no existe
os.makedirs(LANDING_PATH, exist_ok=True)
os.makedirs(BRONZE_PATH, exist_ok=True)
os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(QUARANTINE_PATH, exist_ok=True)

print(f"Directorios configurados en: {PROJECT_ROOT}")

Directorios configurados en: /content/drive/MyDrive/big-data-final


In [34]:
import shutil
from google.colab import userdata

# Credenciales de AstraDB
ASTRA_CLIENT_ID = userdata.get('ASTRA_CLIENT_ID')
ASTRA_CLIENT_SECRET = userdata.get('ASTRA_CLIENT_SECRET')

# Ruta al Secure Connect Bundle (SCB)
SCB_PATH = f"{PROJECT_ROOT}/secure-connect-cloud-analytics.zip"

if os.path.exists(SCB_PATH):
    print(f"Secure Connect Bundle encontrado en: {SCB_PATH}")
    abs_path = os.path.abspath(SCB_PATH)
    SCB_URI = f"file://{abs_path}"
else:
    print(f"No se encuentra el Secure Connect Bundle en {SCB_PATH}")
    raise FileNotFoundError(f"Sube el secure-connect-bundle.zip a {PROJECT_ROOT}")

if 'spark' in locals():
    spark.stop()

Secure Connect Bundle encontrado en: /content/drive/MyDrive/big-data-final/secure-connect-cloud-analytics.zip


In [35]:
# SparkSession
# Configuramos Spark para que descargue automáticamente el conector de Cassandra y utilice el SCB.
from pyspark.sql import SparkSession

SPARK_PACKAGES = "com.datastax.spark:spark-cassandra-connector_2.12:3.5.0"

spark = SparkSession.builder \
    .appName("CloudProviderAnalytics_Colab") \
    .master("local[*]") \
    .config("spark.jars.packages", SPARK_PACKAGES) \
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions") \
    .config("spark.sql.catalog.myCatalog", "com.datastax.spark.connector.datasource.CassandraCatalog") \
    .config("spark.cassandra.connection.config.cloud.path", SCB_URI) \
    .config("spark.cassandra.auth.username", ASTRA_CLIENT_ID) \
    .config("spark.cassandra.auth.password", ASTRA_CLIENT_SECRET) \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"Spark Session iniciada. Versión: {spark.version}")

Spark Session iniciada. Versión: 3.5.0


In [44]:
# Smoke test para verificar la conexión

# 1. Test Spark Local
try:
    print("Test 1: Spark Local DataFrame...")
    spark.range(3).show()
    print("✅ Spark local OK.")
except Exception as e:
    print(f"❌ Error Spark local: {e}")

# 2. Test Conectividad AstraDB
print("\nTest 2: Conexión a AstraDB...")
try:
    # Usamos el catálogo 'myCatalog' que definimos en la configuración
    # Esto le pide a AstraDB la lista de Keyspaces visibles
    spark.sql("SHOW NAMESPACES IN myCatalog").show()
    print("✅ Verifica la existencia de 'cloud_analytics'")
    spark.sql("DESCRIBE NAMESPACE myCatalog.cloud_analytics").show(truncate=False)

except Exception as e:
    print(f"❌ Error al listar bases de datos: {e}")

Test 1: Spark Local DataFrame...
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+

✅ Spark local OK.

Test 2: Conexión a AstraDB...
+------------------+
|         namespace|
+------------------+
|   cloud_analytics|
|data_endpoint_auth|
|      datastax_sla|
+------------------+

✅ Verifica la existencia de 'cloud_analytics'
+--------------+---------------+
|info_name     |info_value     |
+--------------+---------------+
|Catalog Name  |myCatalog      |
|Namespace Name|cloud_analytics|
+--------------+---------------+



## Ingest Batch (Landing -> Bronze)

In [45]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, input_file_name

print("Ingest Batch a Bronze...")

# DEFINICIÓN DE ESQUEMAS
# Definimos los tipos manualmente para asegurar calidad desde el inicio.
# Esto evita que Spark adivine mal (ej. tratar un ID numérico como entero cuando debería ser string)

schemas = {
    "customers_orgs": StructType([
        StructField("org_id", StringType(), True),
        StructField("org_name", StringType(), True),
        StructField("industry", StringType(), True),
        StructField("country", StringType(), True),
        StructField("region", StringType(), True),
        StructField("subscription_plan", StringType(), True),
        StructField("creation_date", DateType(), True) # [cite: 18]
    ]),
    "users": StructType([
        StructField("user_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("email", StringType(), True),
        StructField("role", StringType(), True),
        StructField("status", StringType(), True),
        StructField("last_login", TimestampType(), True) # [cite: 19]
    ]),
    "resources": StructType([
        StructField("resource_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("service", StringType(), True),
        StructField("region", StringType(), True),
        StructField("type", StringType(), True),
        StructField("tags", StringType(), True),
        StructField("creation_time", TimestampType(), True) # [cite: 20]
    ]),
    "billing_monthly": StructType([
        StructField("org_id", StringType(), True),
        StructField("billing_month", StringType(), True), # Formato YYYY-MM
        StructField("currency", StringType(), True),
        StructField("tax_rate", DoubleType(), True),
        StructField("credits_applied", DoubleType(), True),
        StructField("total_due", DoubleType(), True) # [cite: 30]
    ]),
    "support_tickets": StructType([
        StructField("ticket_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("status", StringType(), True),
        StructField("category", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("resolved_at", TimestampType(), True),
        StructField("sla_due", TimestampType(), True),
        StructField("csat_score", IntegerType(), True) # [cite: 21]
    ]),
    "marketing_touches": StructType([
        StructField("touch_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("campaign_id", StringType(), True),
        StructField("channel", StringType(), True),
        StructField("touch_timestamp", TimestampType(), True),
        StructField("converted", BooleanType(), True) # [cite: 22]
    ]),
    "nps_surveys": StructType([
        StructField("survey_id", StringType(), True),
        StructField("org_id", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("survey_date", DateType(), True),
        StructField("score", IntegerType(), True),
        StructField("comment", StringType(), True) # [cite: 29]
    ])
}

# FUNCIÓN DE INGEST ESTÁNDAR
def ingest_batch_file(file_name, schema):
    source_path = f"{LANDING_PATH}/{file_name}.csv"
    dest_path = f"{BRONZE_PATH}/{file_name}"

    print(f"Procesando: {file_name}...")
    try:
        # mode="PERMISSIVE": Si una fila está muy mal formada, pone nulls pero no rompe el proceso
        df = spark.read.csv(source_path, header=True, schema=schema, mode="PERMISSIVE")

        # Agregar marcas de ingesta (Requisito Bronze)
        df_bronze = df \
            .withColumn("ingest_ts", current_timestamp()) \
            .withColumn("source_file", input_file_name())

        # Escribir a Parquet
        df_bronze.write.mode("overwrite").parquet(dest_path)

        count = df_bronze.count()
        print(f"Guardado en Bronze: {dest_path}")
        print(f"Registros procesados: {count}")

    except Exception as e:
        print(f"Error crítico en {file_name}: {e}")

# EJECUCIÓN DEL PIPELINE BATCH
for name, schema in schemas.items():
    ingest_batch_file(name, schema)

print("\nCapa Bronze Batch lista.")

Ingest Batch a Bronze...
Procesando: customers_orgs...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/customers_orgs
Registros procesados: 80
Procesando: users...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/users
Registros procesados: 800
Procesando: resources...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/resources
Registros procesados: 400
Procesando: billing_monthly...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/billing_monthly
Registros procesados: 240
Procesando: support_tickets...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/support_tickets
Registros procesados: 1000
Procesando: marketing_touches...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/marketing_touches
Registros procesados: 1500
Procesando: nps_surveys...
Guardado en Bronze: /content/drive/MyDrive/big-data-final/datalake/bronze/nps_surveys
Registros

In [46]:
print("Verificando tabla 'resources' en Bronze:")
df_check = spark.read.parquet(f"{BRONZE_PATH}/resources")
df_check.show(5)
df_check.printSchema()

Verificando tabla 'resources' en Bronze:
+------------+------------+----------+----------+----------+-------+-------------+--------------------+--------------------+
| resource_id|      org_id|   service|    region|      type|   tags|creation_time|           ingest_ts|         source_file|
+------------+------------+----------+----------+----------+-------+-------------+--------------------+--------------------+
|res_eubfn9kr|org_pnsm43d8|   compute|   sa-east|2025-08-14|running|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_fvb66h3r|org_i7p5tb94|  database|  ap-south|2025-06-05|stopped|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_cbrlqmn4|org_d14ve92m|   storage|eu-central|2025-08-10|running|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_ew1yf0dw|org_pja1wj0t|networking|   us-west|2025-06-02|running|         NULL|2025-12-03 18:50:...|file:///content/d...|
|res_n6mbypjd|org_pja1wj0t|   storage|   us-west|2025-06-05|running|         NULL|20

### Ingest Streaming (Landing -> Bronze)

Los archivos JSONL en `usage_events_stream/` simulan un flujo continuo de eventos. Tienes un reto clave mencionado en el enunciado: Evolución de Esquema.

Al principio, los eventos tienen un esquema V1.

A partir de cierta fecha (~2025-07-18), aparece la `schema_version=2` con campos nuevos: `genai_tokens` y `carbon_kg`.

In [53]:
# Para que el stream no falle cuando aparezcan los campos nuevos,
# definimos un superset, que inicialmente Spark llenara con null

print("Ingest Streaming...")

# ===== PARA DEBUG =====
# Borramos los checkpoints y datos anteriores para evitar conflictos de esquema
if os.path.exists(f"{CHECKPOINT_PATH}/bronze_events"):
    shutil.rmtree(f"{CHECKPOINT_PATH}/bronze_events")
if os.path.exists(f"{BRONZE_PATH}/usage_events"):
    shutil.rmtree(f"{BRONZE_PATH}/usage_events")
print("Checkpoints y datos anteriores eliminados.")
# ======================

# DEFINIR ESQUEMA UNIFICADO (V1 + V2)
# Si un JSON no tiene algun campo, Spark le pone null.
json_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("org_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("service", StringType(), True),
    StructField("region", StringType(), True),
    StructField("resource_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("value", DoubleType(), True),
    StructField("unit", StringType(), True),
    StructField("cost_usd_increment", DoubleType(), True),
    StructField("schema_version", StringType(), True),
    # Campos nuevos de V2 (apareceran como null en V1)
    StructField("genai_tokens", LongType(), True),
    StructField("carbon_kg", DoubleType(), True)
])

# CONFIGURAR EL READER
print("Configurando lectura de stream desde Landing...")

df_stream_raw = spark.readStream \
    .format("json") \
    .schema(json_schema) \
    .option("maxFilesPerTrigger", 10) \
    .load(f"{LANDING_PATH}/usage_events_stream") #

# LÓGICA DE DEDUPE Y METADATA
# Requisito: Deduplicar por event_id y manejar late data
df_stream_bronze = df_stream_raw \
    .withColumnRenamed("timestamp", "event_ts") \
    .withColumn("ingest_ts", current_timestamp()) \
    .withColumn("source_file", input_file_name()) \
    .withWatermark("event_ts", "2 hours") \
    .dropDuplicates(["event_id", "event_ts"])

# CONFIGURAR EL WRITER (A BRONZE PARQUET)
# Guardamos en Parquet particionado por fecha para optimizar lecturas futuras
# Usamos checkpointing en Drive para tolerancia a fallos
bronze_stream_query = df_stream_bronze \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/bronze_events") \
    .option("path", f"{BRONZE_PATH}/usage_events") \
    .partitionBy("service") \
    .trigger(availableNow=True) \
    .start()

print("Stream iniciado con trigger='availableNow'...")
print("- Esto procesará TODOS los archivos históricos disponibles.")
print("- Espera a que termine el proceso...")

# Esperar a que termine de procesar los archivos actuales
bronze_stream_query.awaitTermination()

print("Ingesta Streaming completada (Modo Batch Inicial).")

Ingest Streaming...
Checkpoints y datos anteriores eliminados.
Configurando lectura de stream desde Landing...
Stream iniciado con trigger='availableNow'...
- Esto procesará TODOS los archivos históricos disponibles.
- Espera a que termine el proceso...
Ingesta Streaming completada (Modo Batch Inicial).


In [57]:
from pyspark.sql.functions import col

print("Inspeccionando capa Bronze (Streaming Events):")

try:
    path_events = f"{BRONZE_PATH}/usage_events"
    df_events = spark.read.parquet(path_events)

    total_events = df_events.count()
    print(f"Total de eventos ingestados: {total_events}")
    print("Muestra de datos (incluyendo columnas v2):")
    df_events.select("event_ts", "service", "cost_usd_increment", "genai_tokens", "carbon_kg").show(10)
    print("Servicios encontrados (Particiones):")
    df_events.select("service").distinct().show()

except Exception as e:
    print(f"Aún no hay datos o ruta incorrecta: {e}")

Inspeccionando capa Bronze (Streaming Events):
Total de eventos ingestados: 7245
Muestra de datos (incluyendo columnas v2):
+-------------------+-------+------------------+------------+---------+
|           event_ts|service|cost_usd_increment|genai_tokens|carbon_kg|
+-------------------+-------+------------------+------------+---------+
|2025-08-10 05:05:00|compute|            0.2516|        NULL|  5.38E-4|
|2025-07-30 23:05:00|compute|            6.5886|        NULL|   0.0186|
|2025-08-13 12:54:00|compute|            0.1108|        NULL|  2.74E-4|
|2025-07-11 02:00:00|compute|            0.5922|        NULL|     NULL|
|2025-08-03 01:22:00|compute|            7.7236|        NULL|   0.0226|
|2025-08-13 01:28:00|compute|            1.4459|        NULL|   0.0043|
|2025-07-15 12:52:00|compute|            9.8201|        NULL|     NULL|
|2025-08-18 13:52:00|compute|           11.6419|        NULL|   0.0268|
|2025-07-27 05:44:00|compute|            7.3342|        NULL|     0.02|
|2025-08-15 

## Capa Silver

In [58]:
from pyspark.sql.functions import col, to_date, lower, trim, when, lit, coalesce

print("Procesamiento Silver (Enriquecimiento)...")

# CARGAR DIMENSIONES ESTÁTICAS (Lookups)
# Leemos las tablas maestras de Bronze y las cacheamos en memoria.
print("Cargando dimensiones en memoria...")
try:
    # Organizaciones
    df_orgs = spark.read.parquet(f"{BRONZE_PATH}/customers_orgs") \
        .select("org_id", "org_name", "industry", "subscription_plan") \
        .cache()

    # Usuarios
    df_users = spark.read.parquet(f"{BRONZE_PATH}/users") \
        .select("user_id", "email", "role") \
        .cache()

    # Recursos
    df_resources = spark.read.parquet(f"{BRONZE_PATH}/resources") \
        .select("resource_id", "type", "tags") \
        .cache()

    print(f"- Dimensiones cargadas: Orgs({df_orgs.count()}), Users({df_users.count()})")

except Exception as e:
    print(f"Error cargando dimensiones: {e}")
    raise e

# LEER STREAM DESDE BRONZE
df_bronze_stream = spark.readStream \
    .format("parquet") \
    .schema(spark.read.parquet(f"{BRONZE_PATH}/usage_events").schema) \
    .load(f"{BRONZE_PATH}/usage_events")

# TRANSFORMACIONES SILVER
# Limpieza Básica y Columnas Derivadas
df_clean = df_bronze_stream \
    .withColumn("usage_date", to_date(col("event_ts"))) \
    .withColumn("region", lower(trim(col("region")))) \
    .withColumn("service", lower(trim(col("service")))) \
    .withColumn("unit", when(col("unit").isNull() & col("value").isNotNull(), "count")
    .otherwise(col("unit")))

# Enriquecimiento (Joins)
# Unimos el evento con la info de la organización y el usuario
df_enriched = df_clean \
    .join(df_orgs, "org_id", "left") \
    .join(df_users, "user_id", "left") \
    # No unimos resources para no explotar la memoria si hay millones

# Reglas de Calidad (Valid vs Quarantine)
# Definimos qué es un dato "inválido"
# Costo >= -0.01 se acepta (permitimos pequeños ajustes negativos, pero no errores excesivos)
condicion_valida = (col("cost_usd_increment") >= -0.01) | (col("cost_usd_increment").isNull())

df_silver_valid = df_enriched.filter(condicion_valida)
df_silver_invalid = df_enriched.filter(~condicion_valida) \
    .withColumn("error_reason", lit("Negative Cost Out of Range"))

# ESCRITURA MULTI-STREAM (Valid -> Silver, Invalid -> Quarantine)
print("Iniciando escritura de Streams Silver...")

# Datos Válidos a Silver
query_silver = df_silver_valid \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/silver_main") \
    .option("path", f"{SILVER_PATH}/usage_events_enriched") \
    .partitionBy("usage_date", "service") \
    .trigger(availableNow=True) \
    .start()

# Datos Inválidos a Quarantine
query_quarantine = df_silver_invalid \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/silver_quarantine") \
    .option("path", f"{QUARANTINE_PATH}/usage_events_errors") \
    .trigger(availableNow=True) \
    .start()

print("Procesando enriquecimiento y validación...")
query_silver.awaitTermination()
query_quarantine.awaitTermination()

print("Procesamiento Finalizado: Datos en Silver y Quarantine.")

Procesamiento Silver (Enriquecimiento)...
Cargando dimensiones en memoria...
- Dimensiones cargadas: Orgs(80), Users(800)
Iniciando escritura de Streams Silver...
Procesando enriquecimiento y validación...
Procesamiento Finalizado: Datos en Silver y Quarantine.


In [60]:
print("Inspeccionando Silver (Datos Enriquecidos):")

try:
    df_silver_check = spark.read.parquet(f"{SILVER_PATH}/usage_events_enriched")

    print(f"Total registros Silver: {df_silver_check.count()}")

    print("Muestra con cruces (Joins):")
    df_silver_check.select("event_ts", "org_name", "industry", "service", "cost_usd_increment") \
        .show(10, truncate=False)

    print("Inspeccionando Cuarentena (Errores):")
    # Puede que esté vacío si no había costos negativos, lo cual es bueno
    path_quarantine = f"{QUARANTINE_PATH}/usage_events_errors"
    if os.path.exists(path_quarantine):
        try:
            df_quarantine = spark.read.parquet(path_quarantine)
            if df_quarantine.count() > 0:
                print(f"Se encontraron {df_quarantine.count()} registros inválidos.")
                df_quarantine.select("event_id", "cost_usd_increment", "error_reason").show()
            else:
                print("Carpeta de cuarentena vacía (Sin errores graves).")
        except:
             print("Carpeta de cuarentena vacía o sin archivos parquet aún.")
    else:
        print("No se generó carpeta de cuarentena (Datos limpios).")

except Exception as e:
    print(f"Error leyendo Silver: {e}")

Inspeccionando Silver (Datos Enriquecidos):
Total registros Silver: 7214
Muestra con cruces (Joins):
+-------------------+-----------------+-------------+-------+------------------+
|event_ts           |org_name         |industry     |service|cost_usd_increment|
+-------------------+-----------------+-------------+-------+------------------+
|2025-07-05 23:14:00|Vertex Labs 22   |Manufacturing|compute|11.6565           |
|2025-07-05 12:08:00|Nimbus Digital 36|Education    |compute|5.4476            |
|2025-07-05 02:22:00|Gamma Data 15    |Media        |compute|11.0035           |
|2025-07-05 04:37:00|Delta Tech 71    |Education    |compute|9.9519            |
|2025-07-05 14:22:00|Nova Tech 1      |Education    |compute|0.5091            |
|2025-07-05 08:57:00|Nimbus Cloud 76  |Fintech      |compute|9.4997            |
|2025-07-05 21:05:00|Delta Digital 49 |Fintech      |compute|0.2671            |
|2025-07-05 02:16:00|Zenith Cloud 24  |Manufacturing|compute|1.5218            |
|2025-07